# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a guided example for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In `mlcroissant`, record sets, fields, and columns are key organizational elements referenced by their `@id`s. Let's list all record sets and summarize the fields they contain.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets.")

for rs in record_sets:
    print(f"Record Set [@id]: {rs['@id']}  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"    Field [@id]: {f['@id']}  Name: {f.get('name', 'N/A')}  DataType: {f.get('dataType', 'N/A')}")
    print('---')

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. We reference record sets and fields by their `@id` values below.

For demonstration, we select the principal survey record set. Adjust the record_set_id variable below to match the exact `@id` from the listing above (e.g., 'cr:recordSet/survey').

In [ ]:
# Choose the main survey record set (edit as needed based on your overview output)
main_record_set_id = None
for rs in record_sets:
    if 'survey' in rs.get('name', '').lower() or 'survey' in rs['@id']:
        main_record_set_id = rs['@id']
        break
if not main_record_set_id:
    main_record_set_id = record_sets[0]['@id']  # fallback

# List all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print(f"Record set IDs: {record_set_ids}")

# Load all record sets into DataFrames
dfs = {}
for rset in record_set_ids:
    records = list(dataset.records(record_set=rset))
    if records:
        dfs[rset] = pd.DataFrame(records)

print(f"Columns in record set {main_record_set_id}: {dfs[main_record_set_id].columns.tolist()}")
dfs[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will filter the survey records (referenced by their record set `@id`) on a numeric field, normalize it, and group results based on a key attribute.

**Tip:** Use the field `@id`s, shown in the overview above, for referencing columns.

In [ ]:
# Select a numeric field for analysis (e.g., GAD-7 score)
# Update these variables to match available @id and column name
record_set_id = main_record_set_id
df = dfs[record_set_id]

# Find a numeric column
numeric_field_id = None
for col in df.columns:
    if 'gad7' in col.lower() or 'phq9' in col.lower() or df[col].dtype in [int, float]:
        numeric_field_id = col
        break
if not numeric_field_id:
    numeric_field_id = df.select_dtypes(include=[int, float]).columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filtering records
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical field (e.g., village or education level)
group_field = None
for col in df.columns:
    if 'village' in col.lower() or 'education' in col.lower():
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field}: Mean of {numeric_field_id}")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships.
- Plot the distribution of a numeric score.
- Display grouped means if available.

**Note:** All graphs reference columns using their `@id`.

In [ ]:
# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If group_field exists, plot grouped means
if group_field:
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_field, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrates the use of `mlcroissant` to load, overview, and analyze survey data referencing all entities by their `@id` values.

Key takeaways:
- The FAIR² dataset contains rich demographic and mental health records structured for easy querying.
- All workflows reference dataset entities (record sets, fields, columns) by `@id`, ensuring reproducibility and schema-compatibility.
- Basic EDA and visualizations provide insight into mental health trends and allow community-level analyses.

You may further extend this notebook to incorporate advanced analyses or integrate machine learning models, continuing with entity references using their `@id`s.